# Privacy & Governance Assessment
### NovaCred Credit Application Dataset

This notebook analyzes privacy risks in the NovaCred credit application dataset.

Objectives:
- Identify Personally Identifiable Information (PII)
- Demonstrate pseudonymization / anonymization
- Evaluate GDPR compliance risks
- Propose governance controls


## 1. Dataset Overview

This dataset contains credit application records used by NovaCred's machine learning system.

Each record includes:
- Applicant personal information
- Financial attributes
- Behavioral spending patterns
- Loan approval decision

Because this data is used for automated credit decisions, it falls under GDPR and EU AI Act regulatory scrutiny.



In [2]:
import json
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw_credit_applications.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

type(raw), len(raw)

(list, 502)

In [3]:
# Peek at first record structure
first = raw[0]
print("Top-level keys:", list(first.keys()))
print("\nExample nested keys (applicant_info):", list(first.get("applicant_info", {}).keys()))
print("Example nested keys (financials):", list(first.get("financials", {}).keys()))
print("Example nested keys (decision):", list(first.get("decision", {}).keys()))

Top-level keys: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']

Example nested keys (applicant_info): ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
Example nested keys (financials): ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
Example nested keys (decision): ['loan_approved', 'rejection_reason']


In [4]:
# Flatten nested JSON -> tabular dataframe for auditing
df = pd.json_normalize(raw, sep=".")
df.shape

(502, 21)

In [5]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 80)

df.head(3)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'category': 'Dining', 'amount': 96}, ...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


In [6]:
summary = {
    "records": df.shape[0],
    "columns": df.shape[1],
    "missing_cells": int(df.isna().sum().sum()),
    "missing_cell_rate_pct": round(100 * df.isna().sum().sum() / (df.shape[0] * df.shape[1]), 2),
}
pd.DataFrame([summary])

,records,columns,missing_cells,missing_cell_rate_pct
0,502,21,2619,24.84


In [7]:
missing_by_col = (
    df.isna().mean()
      .sort_values(ascending=False)
      .head(15)
      .mul(100)
      .round(2)
      .reset_index()
)
missing_by_col.columns = ["column", "missing_pct"]
missing_by_col

,column,missing_pct
0,notes,99.60
1,financials.annual_salary,99.00
2,loan_purpose,90.04
3,processing_timestamp,87.65
4,decision.rejection_reason,58.17
5,decision.approved_amount,41.83
6,decision.interest_rate,41.83
7,financials.annual_income,1.00
8,applicant_info.ip_address,1.00
9,applicant_info.ssn,1.00


In [8]:
dtypes_df = df.dtypes.astype(str).value_counts().reset_index()
dtypes_df.columns = ["dtype", "count"]
dtypes_df

,dtype,count
0,object,14
1,float64,4
2,int64,2
3,bool,1


## 2. PII Identification

The first step in privacy analysis is identifying all Personally Identifiable Information (PII) in the dataset.

Examples of PII in this dataset may include:
- Full name
- Email address
- Social Security Number (SSN)
- IP address
- Date of birth

Some attributes (such as ZIP code) may act as quasi-identifiers when combined with other fields.


In [9]:
# Candidate PII / quasi-identifiers (common in credit application datasets)
PII_CANDIDATES = [
    "applicant_info.full_name",
    "applicant_info.name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.ip",
    "applicant_info.date_of_birth",
    "applicant_info.dob",
    "applicant_info.phone",
    "applicant_info.address",
    "applicant_info.zip_code",
    "applicant_info.postal_code",
]

# Keep only those that actually exist in this dataset
pii_cols_found = [c for c in PII_CANDIDATES if c in df.columns]
pii_cols_found

['applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code']

In [10]:
import re

patterns = [
    r"email",
    r"\bssn\b",
    r"ip",
    r"dob",
    r"date.*birth",
    r"birth",
    r"phone",
    r"address",
    r"full_name|fullname|name",
    r"zip|postal",
    r"passport|national_id|id_number",
]

regex = re.compile("|".join(patterns), flags=re.IGNORECASE)

pii_cols_pattern = [c for c in df.columns if regex.search(c)]
pii_cols_pattern[:50], len(pii_cols_pattern)

(['applicant_info.full_name',
  'applicant_info.email',
  'applicant_info.ssn',
  'applicant_info.ip_address',
  'applicant_info.date_of_birth',
  'applicant_info.zip_code'],
 6)

In [11]:
pii_cols = sorted(set(pii_cols_found + pii_cols_pattern))
pii_cols

['applicant_info.date_of_birth',
 'applicant_info.email',
 'applicant_info.full_name',
 'applicant_info.ip_address',
 'applicant_info.ssn',
 'applicant_info.zip_code']

In [12]:
pii_evidence = []
n = len(df)

for col in pii_cols:
    non_null = df[col].notna().sum()
    pii_evidence.append({
        "column": col,
        "non_null_rows": int(non_null),
        "non_null_pct": round(100 * non_null / n, 2),
        "missing_pct": round(100 * (1 - non_null / n), 2),
        "example_values": df[col].dropna().astype(str).head(3).tolist(),
    })

pii_evidence_df = pd.DataFrame(pii_evidence).sort_values("non_null_pct", ascending=False)
pii_evidence_df

,column,non_null_rows,non_null_pct,missing_pct,example_values
1,applicant_info.email,502,100.0,0.0,"[jerry.smith17@hotmail.com, brandon.walker2@yahoo.com, scott.moore94@mail.com]"
2,applicant_info.full_name,502,100.0,0.0,"[Jerry Smith, Brandon Walker, Scott Moore]"
0,applicant_info.date_of_birth,501,99.8,0.2,"[2001-03-09, 1992-03-31, 1989-10-24]"
5,applicant_info.zip_code,501,99.8,0.2,"[10036, 10032, 10075]"
3,applicant_info.ip_address,497,99.0,1.0,"[192.168.48.155, 10.1.102.112, 10.240.193.250]"
4,applicant_info.ssn,497,99.0,1.0,"[596-64-4340, 425-69-4784, 370-78-5178]"


In [13]:
def classify_field(col: str) -> str:
    c = col.lower()
    if "ssn" in c:
        return "PII (direct identifier)"
    if "email" in c:
        return "PII (direct identifier)"
    if "ip" in c:
        return "PII (direct identifier)"
    if "full_name" in c or c.endswith(".name") or c.endswith("name"):
        return "PII (direct identifier)"
    if "date_of_birth" in c or "dob" in c or "birth" in c:
        return "Quasi-identifier (high re-id risk)"
    if "ip_address" in c or c.endswith(".ip"):
        return "PII (direct identifier)"
    return "Review"

pii_inventory = pii_evidence_df[["column", "non_null_pct", "example_values"]].copy()
pii_inventory["category"] = pii_inventory["column"].apply(classify_field)

# Add known sensitive/protected attributes (for governance context)
SENSITIVE_ATTRS = [c for c in df.columns if c.lower().endswith("gender") or ".gender" in c.lower()]
if SENSITIVE_ATTRS:
    extra = pd.DataFrame([{
        "column": c,
        "non_null_pct": round(100 * df[c].notna().mean(), 2),
        "example_values": df[c].dropna().astype(str).head(3).tolist(),
        "category": "Sensitive / Protected attribute (fairness)"
    } for c in sorted(set(SENSITIVE_ATTRS))])
    pii_inventory = pd.concat([pii_inventory, extra], ignore_index=True)

pii_inventory.sort_values(["category", "non_null_pct"], ascending=[True, False]).reset_index(drop=True)

,column,non_null_pct,example_values,category
0,applicant_info.email,100.0,"[jerry.smith17@hotmail.com, brandon.walker2@yahoo.com, scott.moore94@mail.com]",PII (direct identifier)
1,applicant_info.full_name,100.0,"[Jerry Smith, Brandon Walker, Scott Moore]",PII (direct identifier)
2,applicant_info.zip_code,99.8,"[10036, 10032, 10075]",PII (direct identifier)
3,applicant_info.ip_address,99.0,"[192.168.48.155, 10.1.102.112, 10.240.193.250]",PII (direct identifier)
4,applicant_info.ssn,99.0,"[596-64-4340, 425-69-4784, 370-78-5178]",PII (direct identifier)
5,applicant_info.date_of_birth,99.8,"[2001-03-09, 1992-03-31, 1989-10-24]",Quasi-identifier (high re-id risk)
6,applicant_info.gender,99.8,"[Male, M, Male]",Sensitive / Protected attribute (fairness)


### Notes (PII / Quasi-identifiers / Sensitive attributes)

From the privacy audit on 502 applications, we identified multiple personal data fields:

- **Direct identifiers (PII):** full name and email are present in 100% of records; SSN and IP address are present in ~99% of records.
- **Quasi-identifiers:** date of birth and ZIP code are present in ~99.8% of records. Even if not direct identifiers, they can enable re-identification when combined with other attributes.
- **Sensitive / protected attribute (fairness):** gender is present in ~99.8% of records. While not a direct identifier, it is relevant for discrimination risk and must be handled with strict governance controls.

Implication: the raw dataset contains high-risk personal data and requires **data minimization, access control, and de-identification (pseudonymization/anonymization)** before it is used for analytics or ML training.

## 3. PII Inventory Table

We create a table that classifies each sensitive field by:

- Type of identifier
- Privacy risk level
- Necessity for model training
- Recommended treatment



In [14]:
df_priv = df.copy()
df_priv.shape

(502, 21)

In [15]:
# Build final PII inventory table
pii_inventory_table = pii_inventory[[
    "column",
    "category",
    "non_null_pct"
]].copy()

pii_inventory_table = pii_inventory_table.sort_values(
    by=["category", "non_null_pct"],
    ascending=[True, False]
).reset_index(drop=True)

pii_inventory_table

,column,category,non_null_pct
0,applicant_info.email,PII (direct identifier),100.0
1,applicant_info.full_name,PII (direct identifier),100.0
2,applicant_info.zip_code,PII (direct identifier),99.8
3,applicant_info.ip_address,PII (direct identifier),99.0
4,applicant_info.ssn,PII (direct identifier),99.0
5,applicant_info.date_of_birth,Quasi-identifier (high re-id risk),99.8
6,applicant_info.gender,Sensitive / Protected attribute (fairness),99.8


In [16]:
# Add risk level based on category
def risk_level(cat):
    if "direct identifier" in cat.lower():
        return "High"
    if "quasi" in cat.lower():
        return "Medium"
    if "sensitive" in cat.lower():
        return "Medium"
    return "Low"

pii_inventory_table["risk_level"] = pii_inventory_table["category"].apply(risk_level)

pii_inventory_table

,column,category,non_null_pct,risk_level
0,applicant_info.email,PII (direct identifier),100.0,High
1,applicant_info.full_name,PII (direct identifier),100.0,High
2,applicant_info.zip_code,PII (direct identifier),99.8,High
3,applicant_info.ip_address,PII (direct identifier),99.0,High
4,applicant_info.ssn,PII (direct identifier),99.0,High
5,applicant_info.date_of_birth,Quasi-identifier (high re-id risk),99.8,Medium
6,applicant_info.gender,Sensitive / Protected attribute (fairness),99.8,Medium


In [17]:
pii_inventory_table["category"].value_counts()

category
PII (direct identifier)                       5
Quasi-identifier (high re-id risk)            1
Sensitive / Protected attribute (fairness)    1
Name: count, dtype: int64

### PII Inventory Summary

The PII inventory table identifies several personal and sensitive data fields contained in the NovaCred credit application dataset.

The analysis shows that **five attributes are direct identifiers**: email, full name, ZIP code, IP address, and SSN. These fields present a **high privacy risk** because they can directly identify individuals or enable immediate linkage to a specific person.

Additionally, **date of birth** was classified as a **quasi-identifier with high re-identification risk**. Although it does not uniquely identify an individual on its own, it can lead to re-identification when combined with other attributes such as ZIP code or gender.

Finally, the dataset contains the attribute **gender**, which is considered a **sensitive or protected attribute**. While it is not used for direct identification, it is relevant for fairness and discrimination analysis in machine learning models.

Overall, the presence of multiple direct identifiers and quasi-identifiers indicates that the raw dataset poses a **significant privacy risk** if used without proper data protection measures. Therefore, privacy-preserving techniques such as **pseudonymization, anonymization, and strict access controls** should be applied before using the data for analytics or machine learning purposes.

## 4. Pseudonymization Demonstration

To reduce privacy risk, we demonstrate pseudonymization of a PII column.

Example technique:
- Hashing (SHA-256)
- Tokenization

This allows data analysis while protecting direct identifiers.


In [18]:
import hashlib
import os

In [19]:
# Pseudonymization function using SHA-256 hashing

def pseudonymize(value, salt="novacred_demo_salt"):
    if pd.isna(value):
        return value
    
    value_str = str(value).strip().lower()
    
    hashed = hashlib.sha256((salt + value_str).encode("utf-8")).hexdigest()
    
    return hashed

In [20]:
# Create pseudonymized version of email

df_pseudo = df.copy()

df_pseudo["email_pseudo"] = df_pseudo["applicant_info.email"].apply(pseudonymize)

df_pseudo[[
    "applicant_info.email",
    "email_pseudo"
]].head()

,applicant_info.email,email_pseudo
0,jerry.smith17@hotmail.com,43063c5cf90bb5955c56685e16c49ae3286d5e7340d2f48b4038b69190a2dbe6
1,brandon.walker2@yahoo.com,312529264d21756495c2586570d97f51584da56b5fff2b1def4179446ff26e5a
2,scott.moore94@mail.com,3d9144eea942437abb0909e250130915bcf2b5b0c6991e53b95f5e10e949daa9
3,thomas.lee6@protonmail.com,a8ec8411c33ed871c3273bdccfe543685fb9456cede1aaa2ef799399c5bc0672
4,brian.rodriguez86@aol.com,b8da52ca7b20f97eb0837c679d71dc3ed96e59f8f9121139edc423d9468b2834


In [21]:
# Check uniqueness preservation

print("Unique original emails:", df["applicant_info.email"].nunique())
print("Unique pseudonymized emails:", df_pseudo["email_pseudo"].nunique())

Unique original emails: 494
Unique pseudonymized emails: 494


In [22]:
# Remove direct identifier after pseudonymization

df_pseudo = df_pseudo.drop(columns=["applicant_info.email"])

df_pseudo.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,email_pseudo
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790...",2024-01-15T00:00:00Z,Jerry Smith,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,43063c5cf90bb5955c56685e16c49ae3286d5e7340d2f48b4038b69190a2dbe6
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'category': 'Dining', 'amount': 96}, ...",NaN,Brandon Walker,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,312529264d21756495c2586570d97f51584da56b5fff2b1def4179446ff26e5a
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,3d9144eea942437abb0909e250130915bcf2b5b0c6991e53b95f5e10e949daa9
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,a8ec8411c33ed871c3273bdccfe543685fb9456cede1aaa2ef799399c5bc0672
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,b8da52ca7b20f97eb0837c679d71dc3ed96e59f8f9121139edc423d9468b2834


### Pseudonymization Results

To mitigate privacy risks, the direct identifier *email* was pseudonymized using a salted SHA-256 hashing function.

The transformation replaces the original identifier with a cryptographic hash that cannot be easily reversed without knowledge of the salt. This allows analytical operations (such as grouping or counting users) while preventing direct identification of individuals.

The comparison of unique values shows that the number of distinct identifiers remains the same before and after pseudonymization (494 → 494). This confirms that the transformation preserves analytical usefulness while protecting sensitive data.

After generating the pseudonymous identifier, the original email column was removed from the dataset in accordance with **data minimization principles** and privacy-by-design practices.

## 5. Anonymization Techniques

Some fields can be anonymized rather than pseudonymized.

Examples:
- IP address truncation
- Date of birth generalization (age bands)

These methods reduce re-identification risk.


In [24]:
df_anon = df.copy()

df_anon.shape

(502, 21)

In [25]:
# IP address anonymization (truncate last octet)

def mask_ip(ip):
    if pd.isna(ip):
        return ip
    
    parts = str(ip).split(".")
    
    if len(parts) == 4:
        return ".".join(parts[:3]) + ".xxx"
    
    return ip


df_anon["ip_masked"] = df_anon["applicant_info.ip_address"].apply(mask_ip)

df_anon[[
    "applicant_info.ip_address",
    "ip_masked"
]].head()

,applicant_info.ip_address,ip_masked
0,192.168.48.155,192.168.48.xxx
1,10.1.102.112,10.1.102.xxx
2,10.240.193.250,10.240.193.xxx
3,192.168.175.67,192.168.175.xxx
4,172.29.125.105,172.29.125.xxx


In [26]:
# Convert DOB to age

df_anon["dob"] = pd.to_datetime(df_anon["applicant_info.date_of_birth"], errors="coerce")

current_year = 2024

df_anon["age"] = current_year - df_anon["dob"].dt.year

In [27]:
# Age band generalization

def age_band(age):
    if pd.isna(age):
        return None
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    return "55+"


df_anon["age_band"] = df_anon["age"].apply(age_band)

df_anon[[
    "applicant_info.date_of_birth",
    "age",
    "age_band"
]].head()

,applicant_info.date_of_birth,age,age_band
0,2001-03-09,23.0,18-24
1,1992-03-31,32.0,25-34
2,1989-10-24,35.0,35-44
3,1983-04-25,41.0,35-44
4,1999-05-21,25.0,25-34


In [28]:
# Remove exact date of birth

df_anon = df_anon.drop(columns=["applicant_info.date_of_birth"])

df_anon.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,ip_masked,dob,age,age_band
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,192.168.48.xxx,2001-03-09,23.0,18-24
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'category': 'Dining', 'amount': 96}, ...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,10.1.102.xxx,1992-03-31,32.0,25-34
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,10.240.193.xxx,1989-10-24,35.0,35-44
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,192.168.175.xxx,1983-04-25,41.0,35-44
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,172.29.125.xxx,1999-05-21,25.0,25-34


### Anonymization Results

Two anonymization techniques were applied to reduce the risk of re-identification.

First, IP addresses were partially masked by truncating the last octet, preserving general network information while hiding the exact device address.  
Second, dates of birth were generalized into age bands, replacing precise birth dates with broader demographic groups.

These transformations reduce the possibility of identifying individuals while maintaining useful information for analytical purposes.

## 6. GDPR Compliance Assessment

We map the dataset and processing pipeline to key GDPR principles:

- Lawfulness and transparency
- Data minimization
- Storage limitation
- Accountability


### GDPR Compliance Assessment

The analysis highlights several GDPR-relevant risks within the raw credit application dataset. The presence of direct identifiers such as email, full name, IP address, and SSN indicates that the dataset initially contains personally identifiable information (PII) that requires careful governance.

Several GDPR principles were considered in the data processing pipeline:

- **Lawfulness and transparency:** Personal data processing must have a clear legal basis and transparent data usage policies.
- **Data minimization:** Direct identifiers should not be used unless strictly necessary for model training or operational processes.
- **Storage limitation:** Sensitive personal data should not be retained longer than required for legitimate business purposes.
- **Accountability:** Organizations must implement governance processes to ensure responsible handling of personal data.

The pseudonymization and anonymization techniques applied in this notebook demonstrate practical measures that support GDPR compliance and reduce privacy risks in data-driven systems.

## 6.5 Privacy Controls Coverage 

### Privacy Controls Coverage

Based on the analysis, several privacy controls can be implemented to protect sensitive information in the dataset.

**Technical controls**
- Pseudonymization of direct identifiers using cryptographic hashing
- Anonymization techniques such as IP masking and age-band generalization
- Access controls to restrict visibility of sensitive attributes

**Organizational controls**
- Data governance policies for handling personal data
- Documentation of data processing activities
- Regular privacy and compliance audits

These controls help reduce the risk of unauthorized identification while enabling the dataset to remain useful for analytics and machine learning purposes.

## 7. Governance Recommendations

Based on the privacy audit, we recommend governance controls such as:

- Pseudonymization of identifiers
- Data retention policies
- Consent tracking mechanisms
- Audit logs for automated decisions
- Human oversight for high-risk decisions


### Governance Recommendations

Based on the privacy analysis conducted in this notebook, several governance measures should be implemented to ensure responsible use of personal data in the credit application system.

First, direct identifiers such as email and IP address should be pseudonymized before being used in analytical pipelines. This reduces privacy risks while preserving the usefulness of the dataset.

Second, organizations should define **data retention policies** to ensure that personal data is not stored longer than necessary for business or regulatory purposes.

Additionally, **consent tracking mechanisms** should be implemented to guarantee that personal data is processed only when appropriate user consent has been obtained.

To increase accountability, **audit logs** should record automated decision processes, especially when machine learning models are involved in credit approval decisions.

Finally, **human oversight** should be maintained for high-risk decisions, ensuring that automated systems remain aligned with ethical and regulatory requirements.

## 8 EU AI Act: why this system is high-risk + governance implications 
(human oversight, logging, documentation)

### EU AI Act Assessment

The credit application system analyzed in this notebook can be classified as a **high-risk AI system** under the EU AI Act. Automated systems used for creditworthiness evaluation directly affect individuals’ access to financial services, which places them in the high-risk category defined by the regulation.

Because these systems may significantly impact individuals’ economic opportunities, strict governance and oversight mechanisms are required.

Several key governance measures should therefore be implemented:

- **Human oversight:** critical decisions such as credit approval or rejection should allow human review to prevent unfair or incorrect automated outcomes.
- **Logging and traceability:** the system should record model decisions and processing steps to enable auditing and accountability.
- **Documentation:** organizations must maintain detailed technical documentation describing datasets, models, and risk mitigation procedures.

These governance measures help ensure that automated credit decision systems operate in a transparent, accountable, and ethically responsible manner in line with EU AI Act requirements.